# aiayn: multi-seed study runner (Week 8 study, Week 9 IWSLT14 pilot and benchmark)

Runs the study named by `STUDY` unattended (ADR-022, ADR-023, docs/KAGGLE.md §11–12):

1. **Finds every study run** already in an attached input (`aiayn-runs` dataset or a previous
   version's output) and reuses it; finished runs are never retrained.
2. Works through the plan in priority order (main runs → checkpoint and decoding selection on
   valid → ablations → sensitivity → tuned configs). Stops cleanly before the session
   limit; the next session continues.
3. Writes `runs/STUDY.md` + `STUDY.json` (also copied into `REFERENCE_RUN/`); for a pilot
   study also `PILOT.md` (gates, curves, memory, throughput; ADR-023 §8).
4. **Publishes** every study run (slimmed to ~30-60 MB each) to the private dataset `aiayn-runs`.

**Launch with Save Version → Save & Run All (Commit)**, GPU **T4**, Internet **ON**, secrets
`KAGGLE_USERNAME`/`KAGGLE_KEY` attached, and the `aiayn-runs` dataset added as input.
Repeat (same settings, newest `aiayn-runs` attached) until `STUDY.md` says *complete*.

In [ ]:
# 0. Parameters
GITHUB_URL = "https://github.com/mohamed242006/Transformer-from-scratch-with-a-paper-reproduction.git"
GITHUB_BRANCH = "master"
STUDY = "configs/study/week9_pilot.yaml"  # Week 8: configs/study/week8.yaml
REFERENCE_RUN = "iwslt14_pilot_transformer_s1"  # STUDY.md is also stored here (published)
PUBLISH = True
RUNS_DATASET = "aiayn-runs"
SESSION_MINUTES = 690  # 11.5 h of Kaggle's 12 h
RESERVE_MINUTES = 40  # report + upload of up to ~4 GB
WORK = "/kaggle/working"

In [ ]:
# 1. Code and dependencies (torch is never reinstalled)
import glob
import importlib.util
import os
import shutil
import subprocess
import sys
import time

SESSION_START = time.time()
REPO = f"{WORK}/aiayn"
if not os.path.exists(f"{REPO}/pyproject.toml"):
    clone = ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_URL, REPO]
    subprocess.run(clone, check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
needed = {"sentencepiece": "sentencepiece>=0.2.0", "sacrebleu": "sacrebleu>=2.4"}
missing = [spec for module, spec in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacremoses==0.2.0"], check=True)  # pinned: decides IWSLT14 split membership (ADR-023)
sys.path.insert(0, f"{REPO}/src")
head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code: git", head)

In [ ]:
# 2. Kaggle API credentials from notebook secrets (needed only for publishing)
KAGGLE_USER = None
try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USER = secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
except Exception as exc:
    print("publishing disabled (no KAGGLE_USERNAME/KAGGLE_KEY secrets):", type(exc).__name__)
print("publish to dataset:", bool(PUBLISH and KAGGLE_USER), f"({KAGGLE_USER}/{RUNS_DATASET})")
print("attached inputs:", sorted(os.listdir("/kaggle/input")) if os.path.exists("/kaggle/input") else [])

In [ ]:
# 3. Plan preview (what this session will work on)
subprocess.run(
    [sys.executable, "scripts/run_study.py", "--study", STUDY, "--work", f"{WORK}/runs",
     "--input", "/kaggle/input", "--unzip-into", "/tmp/aiayn-restored", "--dry-run"],
    check=True,
)

In [ ]:
# 4. Run the study until done or out of session time (exit 0 = complete, 3 = continue later)
minutes = SESSION_MINUTES - (time.time() - SESSION_START) / 60
cmd = [sys.executable, "-u", "scripts/run_study.py", "--study", STUDY, "--work", f"{WORK}/runs",
       "--input", "/kaggle/input", "--unzip-into", "/tmp/aiayn-restored",
       "--minutes", f"{minutes:.0f}", "--reserve", str(RESERVE_MINUTES)]
STUDY_EXIT = subprocess.run(cmd).returncode
print("study exit code:", STUDY_EXIT, "(0 = complete, 3 = run this notebook again)")

In [ ]:
# 5. Report: runs/STUDY.md + STUDY.json (partial results are marked as such)
out = f"{WORK}/runs/STUDY.md"
report = subprocess.run(
    [sys.executable, "scripts/analyze_study.py", "--study", STUDY, "--work", f"{WORK}/runs",
     "--out", out, "--prepare-data"]
)
if report.returncode == 0 and os.path.isdir(f"{WORK}/runs/{REFERENCE_RUN}"):
    for suffix in (".md", ".json"):
        shutil.copy(out.replace(".md", suffix), f"{WORK}/runs/{REFERENCE_RUN}/STUDY{suffix}")
    print(open(out, encoding="utf-8").read()[:3000])

In [ ]:
# 5b. Pilot gates (Week 9 Phase 1 only): PILOT.md with validation curves, memory,
#     throughput, selected learning rates and the Post-LN go/no-go decision (ADR-023 §8)
PILOT_EXIT = None
if "pilot" in STUDY:
    cmd = [sys.executable, "scripts/pilot_report.py", "--study", STUDY,
           "--work", f"{WORK}/runs", "--out", f"{WORK}/runs/PILOT.md"]
    # Attach the validation notebook output(s): all reports are merged, oldest first, so a
    # later re-validation (e.g. GRU at a smaller micro-batch) overrides per model.
    reports = sorted(
        glob.glob("/kaggle/input/**/validation/report.json", recursive=True), key=os.path.getmtime
    )
    for report in reports:
        cmd += ["--validation-report", report]
    PILOT_EXIT = subprocess.run(cmd).returncode
    print("pilot report exit:", PILOT_EXIT, "(0 = Post-LN GO, 2 = NO-GO, 3 = gates pending)")
    if os.path.isdir(f"{WORK}/runs/{REFERENCE_RUN}"):
        for name in ("PILOT.md", "PILOT.json", "pilot_curves.csv"):
            if os.path.exists(f"{WORK}/runs/{name}"):
                shutil.copy(f"{WORK}/runs/{name}", f"{WORK}/runs/{REFERENCE_RUN}/{name}")

In [ ]:
# 6. Publish every study run to the private dataset
runs = sorted(
    d for d in glob.glob(f"{WORK}/runs/*")
    if os.path.exists(f"{d}/metadata.json") or os.path.exists(f"{d}/FAILED.json")
)
if PUBLISH and KAGGLE_USER and runs:
    cmd = [sys.executable, "scripts/kaggle_publish_runs.py"]
    for run_dir in runs:
        cmd += ["--run", run_dir]
    cmd += ["--user", KAGGLE_USER, "--slug", RUNS_DATASET]
    cmd += ["--message", f"week8 study: {len(runs)} runs, exit {STUDY_EXIT}"]
    attached = glob.glob(f"/kaggle/input/{RUNS_DATASET}")
    if attached:
        cmd += ["--previous", attached[0]]
    result = subprocess.run(cmd)
    print("published" if result.returncode == 0 else f"PUBLISH FAILED ({result.returncode})")
else:
    print("not published: the committed notebook output is the copy of these runs")
print(f"{len(runs)} runs in {WORK}/runs;", "study complete" if STUDY_EXIT == 0 else "run again to continue")